# LifeOS — Qwen 2.5:3B QLoRA Fine-Tuning Execution

This reproducible notebook fine-tunes **Qwen/Qwen2.5-3B-Instruct** using **QLoRA (4-bit NF4)** on LifeOS personal operating system datasets.

### Safety Rules & Architecture Invariants
1. **Zero Dev Machine GPU**: Local development workstation (Intel UHD 620) is never used for training.
2. **Production Decoupling**: Active production model `qwen2.5:3b` is untouched.
3. **Zero Cost**: Built for Google Colab Free Tier T4 GPU (15GB VRAM).
4. **No Fabrication**: Only genuine training loss, optimizer steps, and adapter weights are recorded.

## 1. Environment & Hardware Verification
Verify GPU availability, CUDA compatibility, and VRAM headroom.

In [ ]:
!nvidia-smi

import sys
import shutil
import torch

print("=" * 60)
print("[*] PyTorch Version :", torch.__version__)
print("[*] CUDA Available  :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected! Please change runtime to T4 GPU (Runtime -> Change runtime type).")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
free_vram_gb = round(torch.cuda.mem_get_info(0)[0] / (1024**3), 2)
free_disk_gb = round(shutil.disk_usage(".").free / (1024**3), 2)

print(f"[*] Active GPU      : {gpu_name}")
print(f"[*] Total VRAM      : {vram_gb} GB")
print(f"[*] Free VRAM       : {free_vram_gb} GB")
print(f"[*] Free Disk Space : {free_disk_gb} GB")
print("=" * 60)

if vram_gb < 6.0:
    raise RuntimeError(f"Insufficient VRAM ({vram_gb} GB). Minimum required is 6.0 GB for Qwen 2.5:3B QLoRA.")
print("[SUCCESS] Hardware check passed. Proceeding with dependency installation.")

## 2. Install Pinned Dependencies

In [ ]:
!pip install --upgrade pip
!pip install torch==2.2.2 transformers==4.40.2 datasets==2.19.1 peft==0.10.0 trl==0.8.6 bitsandbytes==0.43.1 accelerate==0.29.3 scipy==1.12.0

## 3. Upload & Verify LifeOS Dataset
Upload your exported LifeOS dataset (`.jsonl`). The cell verifies record counts and SHA-256 integrity.

In [ ]:
import os
import json
import hashlib
from google.colab import files

dataset_filename = "lifeos_training_dataset.jsonl"

if not os.path.exists(dataset_filename):
    print("[*] Please upload your exported LifeOS dataset JSONL file:")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith(".jsonl"):
            dataset_filename = fn
            break

with open(dataset_filename, "rb") as f:
    dataset_sha256 = hashlib.sha256(f.read()).hexdigest()

with open(dataset_filename, "r", encoding="utf-8") as f:
    sample_lines = [l.strip() for l in f if l.strip()]

print(f"[*] Loaded Dataset File : {dataset_filename}")
print(f"[*] Dataset SHA-256     : {dataset_sha256}")
print(f"[*] Total Sample Count  : {len(sample_lines)}")

if len(sample_lines) < 5:
    raise ValueError(f"Sample count ({len(sample_lines)}) is below the technical floor of 5 samples.")
if len(sample_lines) < 100:
    print("[ADVISORY] Dataset has < 100 samples. While meeting technical minimums, larger datasets (100+) are recommended for generalizability.")

## 4. Execute QLoRA 4-bit SFT Training
Configures Qwen 2.5:3B with NF4 quantization, sets up LoRA adapter rank=16, and trains with TRL SFTTrainer.

In [ ]:
import time
import random
from datetime import datetime, timezone
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

BASE_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
SEED = 42
random.seed(SEED)

# Format into ChatML
records = [json.loads(l) for l in sample_lines]
formatted_records = []
for r in records:
    sys_msg = "You are LifeOS, an intelligent, empathetic, and highly disciplined personal operating system assistant."
    if r.get("context"):
        sys_msg += f"\n\nUser Context:\n{r['context']}"
    formatted_records.append({
        "messages": [
            {"role": "system", "content": sys_msg},
            {"role": "user", "content": r.get("prompt", r.get("prompt_input", ""))},
            {"role": "assistant", "content": r.get("response", r.get("chosen_response", ""))}
        ]
    })

# Split 80/20 train/validation
random.shuffle(formatted_records)
val_size = max(1, int(len(formatted_records) * 0.2))
train_ds = Dataset.from_list(formatted_records[val_size:])
val_ds = Dataset.from_list(formatted_records[:val_size])
print(f"[*] Train Dataset Size : {len(train_ds)}")
print(f"[*] Val Dataset Size   : {len(val_ds)}")

# Quantization (4-bit NF4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

# LoRA Adapter Config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, peft_config)
trainable, total = model.get_nb_trainable_parameters()
print(f"[*] Trainable Parameters: {trainable:,} / {total:,} ({100 * trainable / total:.3f}%)")

# Training Arguments
output_dir = "./lifeos_adapter_output"
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    logging_steps=1,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    max_grad_norm=0.3,
    seed=SEED,
    report_to="none",
)

def formatting_prompts_func(example):
    return tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=peft_config,
    formatting_func=formatting_prompts_func,
    max_seq_length=1024,
    tokenizer=tokenizer,
    args=training_args,
)

print("\n[*] Launching Real Training...")
start_t = time.time()
train_res = trainer.train()
duration_sec = round(time.time() - start_t, 2)
print(f"[+] Training Completed in {duration_sec}s")

eval_res = trainer.evaluate()
print(f"[*] Final Train Loss: {train_res.training_loss:.4f}")
print(f"[*] Final Val Loss  : {eval_res.get('eval_loss', 0.0):.4f}")

## 5. Artifact Packaging & SHA-256 Checksum Calculation

In [ ]:
final_adapter_dir = os.path.join(output_dir, "final_adapter")
trainer.save_model(final_adapter_dir)
tokenizer.save_pretrained(final_adapter_dir)

# Compute adapter checksum
hasher = hashlib.sha256()
for root, _, files_list in sorted(os.walk(final_adapter_dir)):
    for fname in sorted(files_list):
        fpath = os.path.join(root, fname)
        hasher.update(fname.encode("utf-8"))
        with open(fpath, "rb") as f:
            while chunk := f.read(65536):
                hasher.update(chunk)
adapter_checksum = hasher.hexdigest()

manifest = {
    "base_model": BASE_MODEL_NAME,
    "dataset_sha256": dataset_sha256,
    "sample_count": len(sample_lines),
    "train_loss": train_res.training_loss,
    "val_loss": eval_res.get("eval_loss"),
    "training_duration_sec": duration_sec,
    "global_steps": train_res.global_step,
    "adapter_checksum": adapter_checksum,
    "completed_at": datetime.now(timezone.utc).isoformat(),
}

manifest_path = os.path.join(output_dir, "training_manifest.json")
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("=" * 60)
print("[+] Artifact Checksum:", adapter_checksum)
print("[+] Manifest Created :", manifest_path)
print("=" * 60)

# Zip for download
!zip -r lifeos_qwen25_3b_adapter.zip ./lifeos_adapter_output